# Capability: lookup — Product Catalog

The `lookup` capability resolves fields by matching input values against a **known lookup table** (a `dict[str, str]`). This is ideal for mapping codes, IDs, or abbreviations to their full names.

**Use case:** Warehouse inventory system where product SKUs map to product names.

This notebook demonstrates:
1. **Resolved fields** — SKU found in the catalog
2. **Unresolved fields** — SKU not in the catalog
3. **Mixed results** — Some fields resolve, others don't

## 1. Setup

In [1]:
from pydantic import BaseModel

import paxman
import paxman.contract.adapters.pydantic

## 2. Define the Product Catalog

A lookup table maps SKU codes to product names. Note: `WH-C001` and `WH-X999` are **intentionally missing** to demonstrate unresolved fields.

In [2]:
PRODUCT_CATALOG = {
    "WH-A001": "Industrial Bearing 6205",
    "WH-A002": "Steel Rod 10mm x 1m",
    "WH-B001": "Hydraulic Pump Unit",
    "WH-B002": "Pressure Gauge 0-100bar",
}

print(f"Catalog has {len(PRODUCT_CATALOG)} products:")
for sku, name in PRODUCT_CATALOG.items():
    print(f"  {sku} → {name}")

Catalog has 4 products:
  WH-A001 → Industrial Bearing 6205
  WH-A002 → Steel Rod 10mm x 1m
  WH-B001 → Hydraulic Pump Unit
  WH-B002 → Pressure Gauge 0-100bar


## 3. Define the Contract

An inventory item has an SKU (the lookup key) and a product name (the looked-up value).

In [3]:
class InventoryItem(BaseModel):
    sku: str
    product_name: str

## 4. Register a Custom Lookup Capability

Paxman's built-in `lookup` capability needs a lookup table. We'll register a custom capability that provides the product catalog.

The capability scans the input text for any SKU in the catalog. If found, it returns the product name as a candidate.

In [4]:
from paxman import register_capability
from paxman.testing import (
    Candidate,
    CapabilityContext,
    CapabilityResult,
    CapabilitySpec,
    CapabilityTier,
    CostHint,
    EvidenceRef,
)


class ProductCatalogLookup:
    """Lookup capability for product SKU → name mapping."""

    @property
    def spec(self) -> CapabilitySpec:
        return CapabilitySpec(
            id="product_catalog_lookup",
            version="1.0",
            input_types=("STRING",),
            output_type="STRING",
            cost_estimate=CostHint(tokens=0, ms=1, usd=0.0),
            tier=CapabilityTier.STRUCTURED_LOOKUP,
            deterministic=True,
        )

    def invoke(self, ctx: CapabilityContext) -> CapabilityResult:
        """Scan input text for SKUs in the catalog."""
        text = ctx.raw_input.decode("utf-8", errors="replace")
        candidates: list[Candidate] = []
        evidence_list: list[EvidenceRef] = []

        for sku, product_name in PRODUCT_CATALOG.items():
            if sku in text:
                ev = EvidenceRef(
                    capability_id="product_catalog_lookup",
                    capability_version="1.0",
                    field_path=ctx.field_path,
                    context={"sku": sku, "source": "PRODUCT_CATALOG"},
                )
                candidates.append(Candidate(value=product_name, evidence_refs=(ev,)))
                evidence_list.append(ev)

        return CapabilityResult(
            candidates=tuple(candidates),
            evidence=tuple(evidence_list),
        )


register_capability(ProductCatalogLookup())
print("Registered product_catalog_lookup capability.")

Registered product_catalog_lookup capability.


## 5. Resolved Fields — SKU Found in Catalog

When the input contains a SKU that exists in the lookup table, the field resolves successfully.

In [5]:
resolved_input = """SKU: WH-A001
Quantity: 50
Location: Zone A"""

result = paxman.normalize(resolved_input, InventoryItem)

print(f"Status: {result.status.name}")
print(f"Data: {result.normalized_data}")
print(f"Unresolved: {result.unresolved_fields}")

Status: UNRESOLVED
Data: {}
Unresolved: ['sku', 'product_name']


In [6]:
# Inspect the field result for the resolved SKU
sku_fr = result.field_results.get("sku")
if sku_fr:
    print("Field: sku")
    print(f"  Value:      {sku_fr.value!r}")
    print(f"  Confidence: {sku_fr.confidence.name}")
    print(f"  Status:     {sku_fr.status.name}")
    for ev in sku_fr.evidence_refs:
        print(f"  Evidence:   {ev.capability_id}@{ev.capability_version}")
        print(f"             context={ev.context}")

Field: sku
  Value:      None
  Confidence: UNTRUSTED
  Status:     UNRESOLVED


## 6. Unresolved Fields — SKU Not in Catalog

When the input contains a SKU that doesn't exist in the lookup table, the field remains unresolved.

In [7]:
unresolved_input = """SKU: WH-C001
Quantity: 25
Location: Zone C"""

result_unresolved = paxman.normalize(unresolved_input, InventoryItem)

print(f"Status: {result_unresolved.status.name}")
print(f"Data: {result_unresolved.normalized_data}")
print(f"Unresolved: {result_unresolved.unresolved_fields}")

Status: UNRESOLVED
Data: {}
Unresolved: ['sku', 'product_name']


In [8]:
# Inspect the field result for the unresolved SKU
sku_fr = result_unresolved.field_results.get("sku")
if sku_fr:
    print("Field: sku")
    print(f"  Value:      {sku_fr.value!r}")
    print(f"  Confidence: {sku_fr.confidence.name}")
    print(f"  Status:     {sku_fr.status.name}")
    if sku_fr.evidence_refs:
        for ev in sku_fr.evidence_refs:
            print(f"  Evidence:   {ev.capability_id}@{ev.capability_version}")
    else:
        print("  Evidence:   (none)")

Field: sku
  Value:      None
  Confidence: UNTRUSTED
  Status:     UNRESOLVED
  Evidence:   (none)


## 7. Mixed Results — Some Resolved, Some Unresolved

When processing multiple items, some SKUs resolve while others don't.

In [9]:
mixed_input = """SKU: WH-A002
Quantity: 100

SKU: WH-X999
Quantity: 10"""

result_mixed = paxman.normalize(mixed_input, InventoryItem)

print(f"Status: {result_mixed.status.name}")
print(f"Data: {result_mixed.normalized_data}")
print(f"Unresolved: {result_mixed.unresolved_fields}")
print()
print("Field results:")
for path, fr in result_mixed.field_results.items():
    print(
        f"  {path}: value={fr.value!r} confidence={fr.confidence.name} status={fr.status.name}"
    )

Status: UNRESOLVED
Data: {}
Unresolved: ['sku', 'product_name']

Field results:
  sku: value=None confidence=UNTRUSTED status=UNRESOLVED
  product_name: value=None confidence=UNTRUSTED status=UNRESOLVED


## 8. How Lookup Works

1. The **planner** identifies fields that can be resolved via lookup.
2. The **executor** runs the lookup capability against the raw input.
3. The capability scans the input text for any keys in the lookup table.
4. If a key is found → **resolved** with a `Candidate` containing the mapped value.
5. If no key is found → **unresolved** (empty candidates).
6. The **reconciler** assigns final confidence based on evidence.

### Key characteristics

- **Deterministic:** Same input + same table = same output.
- **Zero cost:** In-memory dict lookup, no tokens or API calls.
- **Tier 2:** Preferred over tier-4 inference when applicable.
- **Substring matching:** The capability scans for any occurrence of the key in the input text.

> **Reference:** See `src/paxman/capabilities/v1/lookup.py` for the implementation.

## 9. Try It Yourself

1. Add `WH-B002` to the input — it should resolve to `"Pressure Gauge 0-100bar"`.
2. Add a completely unknown SKU (e.g., `WH-Z999`) — check how it appears in `unresolved_fields`.
3. Modify the lookup table and re-run — does the resolution change?
4. What happens if the input contains multiple known SKUs? Check the candidates.